# Phase 1 — Exploratory Data Analysis
## Online Retail II Dataset

15 charts, each answering a specific business question about the retail dataset.
Every chart has a markdown cell above it stating the question it answers.

## Setup — Imports & Pipeline Run

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from statsmodels.tsa.seasonal import seasonal_decompose
import warnings
warnings.filterwarnings('ignore')

from models.data_pipeline import (
    load_raw_data, extract_signal_before_cleaning,
    clean_data, calculate_rfm, engineer_features,
    build_cohort_matrix, build_master_customer_table
)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.titlesize'] = 13

In [ ]:
df_raw = load_raw_data()
return_features, cancel_features = extract_signal_before_cleaning(df_raw)
df_customers, df_all = clean_data(df_raw)
rfm = calculate_rfm(df_customers)
master = build_master_customer_table(df_customers, return_features, cancel_features)
retention_matrix, cohort_size = build_cohort_matrix(df_customers)

print(f'customers: {master.shape[0]:,}  |  features: {master.shape[1]}')

---
## Chart 1 — Revenue Distribution (Log Scale)
**Business question: Is revenue power-law distributed?**

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
revenue_pos = df_customers[df_customers['revenue'] > 0]['revenue']
ax.hist(revenue_pos, bins=200, color='steelblue', edgecolor='none', alpha=0.85)
ax.set_yscale('log')
ax.set_xscale('log')
ax.set_title('Revenue per Transaction — Log-Log Scale')
ax.set_xlabel('Revenue (£) — log scale')
ax.set_ylabel('Count — log scale')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x:,.0f}'))
plt.tight_layout()
plt.show()

print(f'median revenue : £{revenue_pos.median():.2f}')
print(f'mean revenue   : £{revenue_pos.mean():.2f}')
print(f'99th pct       : £{revenue_pos.quantile(0.99):.2f}')

---
## Chart 2 — Pareto: Top 20% Customers vs Bottom 80% Revenue
**Business question: How concentrated is revenue across the customer base?**

In [ ]:
sorted_rev = master[['customer_id', 'total_revenue']].sort_values('total_revenue', ascending=False).reset_index(drop=True)
sorted_rev['cum_rev_pct'] = sorted_rev['total_revenue'].cumsum() / sorted_rev['total_revenue'].sum() * 100
sorted_rev['customer_pct'] = (sorted_rev.index + 1) / len(sorted_rev) * 100

top20_rev = sorted_rev[sorted_rev['customer_pct'] <= 20]['total_revenue'].sum()
bot80_rev = sorted_rev[sorted_rev['customer_pct'] > 20]['total_revenue'].sum()
total_rev = sorted_rev['total_revenue'].sum()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pareto bar
axes[0].bar(['Top 20% customers', 'Bottom 80% customers'],
            [top20_rev / total_rev * 100, bot80_rev / total_rev * 100],
            color=['steelblue', 'lightcoral'])
axes[0].set_ylabel('% of Total Revenue')
axes[0].set_title('Revenue Share: Top 20% vs Bottom 80% Customers')
for i, v in enumerate([top20_rev / total_rev * 100, bot80_rev / total_rev * 100]):
    axes[0].text(i, v + 0.5, f'{v:.1f}%', ha='center', fontweight='bold')

# Lorenz-style cumulative curve
axes[1].plot(sorted_rev['customer_pct'], sorted_rev['cum_rev_pct'], color='steelblue')
axes[1].plot([0, 100], [0, 100], 'k--', alpha=0.4, label='Perfect equality')
axes[1].axvline(20, color='red', linestyle='--', alpha=0.6, label='Top 20%')
axes[1].set_xlabel('Cumulative % of Customers')
axes[1].set_ylabel('Cumulative % of Revenue')
axes[1].set_title('Lorenz Curve — Revenue Concentration')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## Chart 3 — Monthly Revenue Line Chart with Trend
**Business question: What is the revenue trajectory over time?**

In [ ]:
monthly = (
    df_customers
    .assign(month=df_customers['invoice_date'].dt.to_period('M'))
    .groupby('month')['revenue'].sum()
    .reset_index()
)
monthly['month_dt'] = monthly['month'].dt.to_timestamp()
monthly['trend'] = monthly['revenue'].rolling(window=3, center=True).mean()

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(monthly['month_dt'], monthly['revenue'], marker='o', color='steelblue',
        linewidth=1.5, markersize=4, label='Monthly revenue')
ax.plot(monthly['month_dt'], monthly['trend'], color='red',
        linewidth=2.5, linestyle='--', label='3-month rolling average')
ax.set_title('Monthly Revenue with 3-Month Rolling Trend')
ax.set_xlabel('Month')
ax.set_ylabel('Revenue (£)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x/1e6:.1f}M'))
ax.legend()
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

---
## Chart 4 — Cohort Retention Heatmap
**Business question: When do customers drop off after their first purchase?**

In [ ]:
ret_plot = retention_matrix.iloc[:, :13].copy()
ret_plot.index = ret_plot.index.astype(str)

plt.figure(figsize=(18, 9))
sns.heatmap(
    ret_plot,
    annot=True, fmt='.0%',
    cmap='YlGnBu',
    linewidths=0.5,
    vmin=0, vmax=1,
    cbar_kws={'label': 'Retention Rate'}
)
plt.title('Monthly Cohort Retention Rate (first 12 months after acquisition)', fontsize=14)
plt.xlabel('Months Since First Purchase')
plt.ylabel('Acquisition Cohort')
plt.tight_layout()
plt.show()

---
## Chart 5 — RFM Scatter: Recency vs Frequency, Bubble = Monetary
**Business question: How is the RFM space distributed across customers?**

In [ ]:
rfm_plot = master.copy()
rfm_plot['monetary_clip'] = rfm_plot['monetary'].clip(upper=rfm_plot['monetary'].quantile(0.97))
rfm_plot['frequency_clip'] = rfm_plot['frequency'].clip(upper=rfm_plot['frequency'].quantile(0.97))
rfm_plot['recency_clip'] = rfm_plot['recency'].clip(upper=rfm_plot['recency'].quantile(0.97))

fig = px.scatter(
    rfm_plot.sample(min(3000, len(rfm_plot)), random_state=42),
    x='recency_clip', y='frequency_clip',
    size='monetary_clip', color='monetary_clip',
    color_continuous_scale='Viridis',
    size_max=20, opacity=0.6,
    labels={
        'recency_clip': 'Recency (days since last purchase)',
        'frequency_clip': 'Frequency (unique orders)',
        'monetary_clip': 'Monetary (mean order £)'
    },
    title='RFM Scatter — Recency vs Frequency (bubble size & colour = Monetary)'
)
fig.show()

---
## Chart 6 — Revenue by Country (Top 10)
**Business question: Which geographies drive the most revenue?**

In [ ]:
country_rev = (
    df_customers.groupby('country')['revenue']
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.barh(country_rev['country'][::-1], country_rev['revenue'][::-1], color='steelblue')
ax.set_xlabel('Total Revenue (£)')
ax.set_title('Top 10 Countries by Revenue')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x/1e6:.1f}M'))
for bar, val in zip(bars, country_rev['revenue'][::-1]):
    ax.text(val * 1.01, bar.get_y() + bar.get_height() / 2,
            f'£{val/1e6:.2f}M', va='center', fontsize=9)
plt.tight_layout()
plt.show()

---
## Chart 7 — Return Rate Distribution
**Business question: What does the customer return rate distribution look like?**

In [ ]:
rr = master[master['return_rate'] > 0]['return_rate']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(rr.clip(upper=rr.quantile(0.99)), bins=60, color='coral', edgecolor='white')
axes[0].set_title('Return Rate Distribution (customers with ≥1 return)')
axes[0].set_xlabel('Return Rate')
axes[0].set_ylabel('Number of Customers')

axes[1].boxplot(rr.clip(upper=rr.quantile(0.99)), vert=False, patch_artist=True,
                boxprops=dict(facecolor='coral', alpha=0.6))
axes[1].set_title('Return Rate Box Plot')
axes[1].set_xlabel('Return Rate')

plt.suptitle(f'{(master["return_rate"] > 0).sum():,} of {len(master):,} customers have at least one return', y=1.01)
plt.tight_layout()
plt.show()

print(rr.describe().round(4))

---
## Chart 8 — Cancellation Rate Distribution
**Business question: How prevalent is order cancellation across customers?**

In [ ]:
cr = master[master['cancellation_rate'] > 0]['cancellation_rate']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(cr.clip(upper=cr.quantile(0.99)), bins=60, color='mediumpurple', edgecolor='white')
axes[0].set_title('Cancellation Rate Distribution (customers with ≥1 cancellation)')
axes[0].set_xlabel('Cancellation Rate')
axes[0].set_ylabel('Number of Customers')

axes[1].boxplot(cr.clip(upper=cr.quantile(0.99)), vert=False, patch_artist=True,
                boxprops=dict(facecolor='mediumpurple', alpha=0.6))
axes[1].set_title('Cancellation Rate Box Plot')
axes[1].set_xlabel('Cancellation Rate')

plt.suptitle(f'{(master["cancellation_rate"] > 0).sum():,} of {len(master):,} customers have at least one cancellation', y=1.01)
plt.tight_layout()
plt.show()

print(cr.describe().round(4))

---
## Chart 9 — Category HHI Distribution
**Business question: How loyal are customers to specific product categories?**

In [ ]:
hhi = master['category_hhi']

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(hhi, bins=50, color='seagreen', edgecolor='white', alpha=0.85)
ax.axvline(hhi.mean(), color='red', linestyle='--', linewidth=1.5, label=f'Mean HHI = {hhi.mean():.2f}')
ax.axvline(hhi.median(), color='orange', linestyle='--', linewidth=1.5, label=f'Median HHI = {hhi.median():.2f}')
ax.set_title('Category HHI Distribution (0 = diverse, 1 = single category)')
ax.set_xlabel('HHI Score')
ax.set_ylabel('Number of Customers')
ax.legend()
plt.tight_layout()
plt.show()

single_cat_pct = (hhi == 1.0).sum() / len(hhi) * 100
print(f'{single_cat_pct:.1f}% of customers buy exclusively from one category (HHI = 1.0)')

---
## Chart 10 — Velocity Decay Ratio Distribution
**Business question: What fraction of customers are showing decelerating purchase behaviour?**

In [ ]:
vdr = master['velocity_decay_ratio'].clip(upper=master['velocity_decay_ratio'].quantile(0.98))

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(vdr, bins=60, color='steelblue', edgecolor='white', alpha=0.85)
ax.axvline(1.0, color='red', linestyle='--', linewidth=2, label='VDR = 1.0 (no change)')
ax.set_title('Velocity Decay Ratio Distribution')
ax.set_xlabel('Velocity Decay Ratio  (>1 = gaps getting longer = decelerating)')
ax.set_ylabel('Number of Customers')
ax.legend()
plt.tight_layout()
plt.show()

decelerating = (master['velocity_decay_ratio'] > 1.0).sum()
pct = decelerating / len(master) * 100
print(f'{decelerating:,} customers ({pct:.1f}%) show a decelerating purchase velocity (VDR > 1)')

---
## Chart 11 — Revenue Seasonality Decomposition
**Business question: What are the trend, seasonal, and residual components of revenue?**

In [ ]:
weekly_rev = (
    df_customers
    .set_index('invoice_date')
    .resample('W')['revenue']
    .sum()
    .dropna()
)

# need at least 2 full periods — period=52 for weekly seasonality
decomp = seasonal_decompose(weekly_rev, model='additive', period=52, extrapolate_trend='freq')

fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)

axes[0].plot(weekly_rev.index, weekly_rev.values, color='steelblue', linewidth=1)
axes[0].set_ylabel('Observed')
axes[0].set_title('Weekly Revenue Seasonality Decomposition (Additive, period=52 weeks)')

axes[1].plot(decomp.trend.index, decomp.trend.values, color='darkorange', linewidth=1.5)
axes[1].set_ylabel('Trend')

axes[2].plot(decomp.seasonal.index, decomp.seasonal.values, color='seagreen', linewidth=1)
axes[2].set_ylabel('Seasonal')

axes[3].plot(decomp.resid.index, decomp.resid.values, color='gray', linewidth=1, alpha=0.7)
axes[3].axhline(0, color='black', linestyle='--', linewidth=0.8)
axes[3].set_ylabel('Residual')
axes[3].set_xlabel('Date')

for ax in axes:
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x/1e3:.0f}k' if abs(x) >= 1000 else f'£{x:.0f}'))

plt.tight_layout()
plt.show()

---
## Chart 12 — Average Order Value Box Plot by Country (Top 10)
**Business question: Do average order values differ meaningfully across geographies?**

In [ ]:
top10_countries = (
    df_customers.groupby('country')['customer_id']
    .nunique()
    .sort_values(ascending=False)
    .head(10)
    .index.tolist()
)

aov_data = (
    df_customers[df_customers['country'].isin(top10_countries)]
    .groupby(['country', 'invoice_no'])['revenue']
    .sum()
    .reset_index()
    .rename(columns={'revenue': 'order_value'})
)

order_for_plot = (
    aov_data.groupby('country')['order_value']
    .median()
    .sort_values(ascending=False)
    .index.tolist()
)

fig, ax = plt.subplots(figsize=(13, 6))
aov_data_clipped = aov_data.copy()
aov_data_clipped['order_value'] = aov_data_clipped.groupby('country')['order_value'].transform(
    lambda x: x.clip(upper=x.quantile(0.95))
)
aov_data_clipped[aov_data_clipped['country'].isin(order_for_plot)].boxplot(
    column='order_value', by='country', ax=ax,
    order=order_for_plot, vert=True, patch_artist=True
)
ax.set_title('Average Order Value by Country — Top 10 by Customer Count (clipped at 95th pct)')
ax.set_xlabel('Country')
ax.set_ylabel('Order Value (£)')
plt.suptitle('')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

---
## Chart 13 — Customer Tenure vs Total Revenue Scatter
**Business question: Does how long a customer has been around predict their lifetime revenue?**

In [ ]:
tenure_plot = master.copy()
tenure_plot['total_revenue_clip'] = tenure_plot['total_revenue'].clip(upper=tenure_plot['total_revenue'].quantile(0.97))

fig = px.scatter(
    tenure_plot.sample(min(3000, len(tenure_plot)), random_state=42),
    x='customer_tenure_days',
    y='total_revenue_clip',
    color='frequency',
    color_continuous_scale='Blues',
    opacity=0.5,
    trendline='ols',
    labels={
        'customer_tenure_days': 'Customer Tenure (days)',
        'total_revenue_clip': 'Total Revenue (£)',
        'frequency': 'Order Frequency'
    },
    title='Customer Tenure vs Total Revenue (coloured by order frequency)'
)
fig.show()

corr = tenure_plot[['customer_tenure_days', 'total_revenue']].corr().iloc[0, 1]
print(f'Pearson correlation (tenure vs total revenue): {corr:.3f}')

---
## Chart 14 — Day-of-Week Revenue Bar
**Business question: Are there consistent weekly revenue patterns?**

In [ ]:
df_dow = df_customers.copy()
df_dow['day_of_week'] = df_dow['invoice_date'].dt.day_name()
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

dow_rev = (
    df_dow.groupby('day_of_week')['revenue']
    .sum()
    .reindex(day_order)
    .reset_index()
)
dow_rev.columns = ['day', 'revenue']

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['steelblue' if d not in ['Saturday', 'Sunday'] else 'lightcoral' for d in dow_rev['day']]
bars = ax.bar(dow_rev['day'], dow_rev['revenue'], color=colors, edgecolor='white')
ax.set_title('Total Revenue by Day of Week')
ax.set_ylabel('Total Revenue (£)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x/1e6:.1f}M'))
for bar, val in zip(bars, dow_rev['revenue']):
    ax.text(bar.get_x() + bar.get_width() / 2, val + val * 0.01,
            f'£{val/1e6:.1f}M', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

---
## Chart 15 — Top 20 Products by Revenue
**Business question: Which individual products drive the most revenue?**

In [ ]:
top_products = (
    df_customers.groupby(['stock_code', 'description'])['revenue']
    .sum()
    .sort_values(ascending=False)
    .head(20)
    .reset_index()
)
top_products['label'] = top_products['stock_code'] + ' — ' + top_products['description'].str[:30]

fig, ax = plt.subplots(figsize=(12, 8))
ax.barh(top_products['label'][::-1], top_products['revenue'][::-1], color='steelblue')
ax.set_title('Top 20 Products by Total Revenue')
ax.set_xlabel('Total Revenue (£)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x/1e3:.0f}k'))
plt.tight_layout()
plt.show()

top20_share = top_products['revenue'].sum() / df_customers['revenue'].sum() * 100
print(f'Top 20 products account for {top20_share:.1f}% of total revenue')

---
## Phase 1 Summary

In [ ]:
print('=== Phase 1 Complete ===')
print(f'unique customers  : {master["customer_id"].nunique():,}')
print(f'feature count     : {master.shape[1]}')
print('\nmedian RFM values:')
print(master[['recency', 'frequency', 'monetary']].median().round(2))
print('\nfeature columns:')
print(list(master.columns))